# 08b: Unseen generators (E2), CLIP trained on SD15 only

Reuses the CLIP features saved by notebook 06 (no GPU needed). The classifier is trained and tuned on SD15 and real images only; the test set contains all six generators.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, log_loss, roc_auc_score
from transformers import CLIPVisionModelWithProjection

ROOT = Path("/content/drive/MyDrive/minor-project")
LOCAL = Path("/content/processed")
FEATURES = ROOT / "features"
FEATURES.mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
RUN_NAME = "e2_m2_sd15"
CLIP_NAME = "openai/clip-vit-large-patch14"
TRAIN_SOURCES = ["Real", "SD15"]
BATCH_SIZE = 128
SEED = 42

## Data

In [ ]:
LOCAL.mkdir(exist_ok=True)
for split in ["train", "val", "test"]:
    for suffix in ["_images.npy", "_meta.csv"]:
        name = split + suffix
        if not (LOCAL / name).exists():
            shutil.copy(ROOT / "processed" / name, LOCAL / name)


def load_split(split):
    images = np.load(LOCAL / f"{split}_images.npy", mmap_mode="r")
    meta = pd.read_csv(LOCAL / f"{split}_meta.csv")
    meta["y"] = (meta["label"] == "fake").astype(int)
    return images, meta


data = {split: load_split(split) for split in ["train", "val", "test"]}

## Extract CLIP features (once)

In [ ]:
tag = CLIP_NAME.split("/")[-1]
needs_clip = any(not (FEATURES / f"{tag}_{split}.npy").exists() for split in data)

dtype = torch.float16 if device.type == "cuda" else torch.float32
if needs_clip:
    clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME, torch_dtype=dtype).to(device).eval()
    IMAGE_SIZE = clip.config.image_size
    print(f"{CLIP_NAME}: {sum(p.numel() for p in clip.parameters()) / 1e6:.0f}M parameters, all frozen")
else:
    print("Using saved CLIP features")
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


@torch.no_grad()
def clip_features(images):
    out = []
    for start in range(0, len(images), BATCH_SIZE):
        batch = torch.from_numpy(np.array(images[start:start + BATCH_SIZE])).to(device)
        x = batch.permute(0, 3, 1, 2).float() / 255
        if x.shape[-1] != IMAGE_SIZE:
            x = F.interpolate(x, size=IMAGE_SIZE, mode="bicubic", align_corners=False)
        x = ((x - CLIP_MEAN) / CLIP_STD).to(dtype)
        emb = clip(pixel_values=x).image_embeds.float()
        out.append(F.normalize(emb, dim=1).cpu().numpy())
    return np.concatenate(out)


feats = {}
for split, (images, meta) in data.items():
    path = FEATURES / f"{tag}_{split}.npy"
    if not path.exists():
        np.save(path, clip_features(images))
    feats[split] = np.load(path)
    print(f"{split}: features {feats[split].shape}")

## Train the classifier

In [ ]:
def keep_sources(meta, sources):
    if sources == "all":
        return meta
    chosen = meta[meta["generator"].isin(sources)]
    fakes = chosen[chosen["y"] == 1]
    reals = chosen[chosen["y"] == 0].sample(n=len(fakes), random_state=SEED)
    return pd.concat([reals, fakes])


def scores(labels, probs):
    return {
        "loss": float(log_loss(labels, probs, labels=[0, 1])),
        "accuracy": float(((probs >= 0.5) == labels).mean()),
        "auc": float(roc_auc_score(labels, probs)),
        "ap": float(average_precision_score(labels, probs)),
    }


train_meta = keep_sources(data["train"][1], TRAIN_SOURCES)
val_meta = keep_sources(data["val"][1], TRAIN_SOURCES)
X_train, y_train = feats["train"][train_meta.index], train_meta["y"].to_numpy()
X_val, y_val = feats["val"][val_meta.index], val_meta["y"].to_numpy()

search = []
for C in [0.01, 0.1, 1, 10, 100]:
    clf = LogisticRegression(C=C, max_iter=5000)
    clf.fit(X_train, y_train)
    val = scores(y_val, clf.predict_proba(X_val)[:, 1])
    search.append({"C": C, **{f"val_{k}": v for k, v in val.items()}})
    print(f"C={C:<6} val loss {val['loss']:.4f} | val acc {val['accuracy']:.3f} | val AUC {val['auc']:.4f}")

best_C = min(search, key=lambda r: r["val_loss"])["C"]
clf = LogisticRegression(C=best_C, max_iter=5000).fit(X_train, y_train)
print(f"Best C = {best_C}; trained parameters: {clf.coef_.size + clf.intercept_.size}")

## Test (once)

In [ ]:
test_meta = data["test"][1]
test_probs = clf.predict_proba(feats["test"])[:, 1]
overall = scores(test_meta["y"].to_numpy(), test_probs)

test_out = test_meta.copy()
test_out["prob_fake"] = test_probs
test_out["correct"] = (test_out["prob_fake"] >= 0.5).astype(int) == test_out["y"]
per_source = test_out.groupby("generator")["correct"].mean()

print("Test:", {k: round(v, 4) for k, v in overall.items()})
display(per_source.round(3).to_frame("accuracy"))

results = {
    "run": RUN_NAME,
    "clip": CLIP_NAME,
    "train_sources": TRAIN_SOURCES,
    "best_C": best_C,
    "test": overall,
    "test_per_source_accuracy": per_source.round(4).to_dict(),
    "search": search,
}
with open(ROOT / "results" / f"{RUN_NAME}.json", "w") as f:
    json.dump(results, f, indent=2)
test_out.to_csv(ROOT / "results" / f"{RUN_NAME}_test_predictions.csv", index=False)
print("Saved results and predictions.")